# 02 · Entrenamiento: barrido de LR y 2 modelos × 3 semillas

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tomas-pucv/Camanchaca-Predict/blob/main/notebooks/02_train.ipynb)

**Propósito.** Entrenar ResNet-50 y DeiT-S con el **mismo protocolo** (`configs/protocol/base.yaml`):
barrido de LR del backbone (criterio en val, desempate: menor LR) y luego 3 semillas por modelo.

**Tiempo esperado (T4).** Barrido ~50 min; `train-all` ~2,5 h (~22 min por corrida). Se puede
repartir en varias sesiones: las corridas terminadas (`DONE`) se omiten y las interrumpidas se
reanudan desde `last.pt`, que vive en Drive.
**Entorno.** GPU T4. Sin GPU, `sweep`/`train-all` se niegan a entrenar: cambie el entorno a
GPU T4 (o pase `--device cpu` a propósito, solo para pruebas).

**Espacio en Drive.** `last.pt` (~280 MB) se reescribe cada `schedule.save_last_every`
evaluaciones (3) y Drive manda cada versión reemplazada a la Papelera, que sigue ocupando
cuota: vacíe la papelera (drive.google.com > Papelera > Vaciar papelera) entre sesiones y al terminar.

**Coherencia.** `selected_lr.json` guarda el hash del protocolo con que se eligió cada LR
(`sweep_protocol_hash`): si cambia `configs/protocol/base.yaml`, `train-all` se niega a usar ese LR
y hay que repetir el barrido. Tampoco permite semillas del mismo modelo con LR distinto.

**Salidas.** `runs/main/sweep/selected_lr.json`; `runs/main/<modelo>/seed<k>/{best.pt, history.csv,
val_metrics.json, config.yaml, DONE}`; `reports/tables/lr_sweep.md`.

## Configuración común
Misma celda en todos los notebooks (detalles en `00_setup_colab`). En **Colab**: monta Drive,
fija `CAMANCHACA_DATA=/content/data` y `CAMANCHACA_RUNS=/content/drive/MyDrive/camanchaca/runs`,
clona/actualiza el repo en la referencia `REF`, instala el paquete (`pip install -e ".[demo,data]"`,
**sin** reinstalar torch) y desempaqueta los `.tar` de datos de Drive (salvo `NEEDS_DATA = False`). **Fuera de Colab** (`SMOKE=True`)
usa datos sintéticos y pesos aleatorios generados por `camanchaca smoke`: sirve para verificar que
todo corre, **sus números no son resultados**.

In [ ]:
# Configuración común (idéntica en los notebooks 00-06)
import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
SMOKE = not IN_COLAB   # True: datos sintéticos + pesos aleatorios (solo verifica que todo corre)
REF = "main"           # rama/tag/commit del repo; para la entrega final usar "v1.0-final"
REPO_URL = "https://github.com/Tomas-pucv/Camanchaca-Predict.git"
DRIVE_DIR = Path("/content/drive/MyDrive/camanchaca")   # tars de datos, corridas, pesos y credenciales

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    os.environ["CAMANCHACA_DATA"] = "/content/data"
    os.environ["CAMANCHACA_RUNS"] = str(DRIVE_DIR / "runs")
    REPO = Path("/content/Camanchaca-Predict")
    if not REPO.exists():
        !git clone -q {REPO_URL} {REPO}
    !git -C {REPO} fetch -q --all --tags && git -C {REPO} checkout -q {REF} && (git -C {REPO} pull -q --ff-only 2>/dev/null || true)
    %cd {REPO}
    !pip install -q -e ".[demo,data]"   # NO reinstala torch/torchvision: se usan los de Colab
    if not SMOKE and globals().get("NEEDS_DATA", True):
        for tar in sorted((DRIVE_DIR / "data").glob("*.tar")):   # imágenes preprocesadas (notebook 01)
            !camanchaca unpack --tar {tar}
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    %cd {REPO}
    if shutil.which("camanchaca") is None:
        print("AVISO: falta el comando `camanchaca`: ejecutar  pip install -e '.[dev,demo]'")

WORK = Path(os.environ.get("CAMANCHACA_SMOKE_DIR", str(REPO / "local_workspace" / "nb_smoke")))
if SMOKE:
    os.environ.update(CAMANCHACA_DATA=str(WORK / "data"), CAMANCHACA_RUNS=str(WORK / "runs"),
                      CAMANCHACA_REPORTS=str(WORK / "reports"), HF_HUB_OFFLINE="1")
    if not (WORK / "smoke.yaml").exists():   # ~4-6 min en CPU, una sola vez
        !camanchaca smoke --workdir {WORK}

sys.path.insert(0, str(REPO / "src"))   # importable sin reiniciar el kernel tras `pip install -e`
from camanchaca.config import load_yaml
from camanchaca.paths import data_root, reports_dir, runs_root

EXP = str(WORK / "smoke.yaml") if SMOKE else "configs/experiment/main.yaml"
EXP_NAME = load_yaml(EXP)["name"]
print(f"Colab={IN_COLAB} | SMOKE={SMOKE} | EXP={EXP}")
print(f"datos={data_root()} | corridas={runs_root()} | reportes={reports_dir()}")

In [ ]:
if IN_COLAB:
    !{sys.executable} scripts/verify_env.py --require-gpu

## 1. Barrido de LR (solo val)

In [ ]:
!camanchaca sweep --exp {EXP}
print((runs_root() / EXP_NAME / "sweep" / "selected_lr.json").read_text())

## 2. Entrenamiento final
Re-ejecutar esta celda tras cualquier desconexión: continúa donde quedó.

In [ ]:
!camanchaca train-all --exp {EXP} --lr auto

## 3. Curvas de aprendizaje y métricas de validación

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
COL = "val_log10_mae_macro_source"
HIST = {f"{p.parent.parent.name}/{p.parent.name}": pd.read_csv(p)
        for p in sorted((runs_root() / EXP_NAME).glob("*/seed*/history.csv"))}
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for name, h in HIST.items(): axes[0].plot(h["step"], h["train_loss"], label=name, alpha=0.8)
for name, h in HIST.items(): axes[1].plot(*h.dropna(subset=[COL])[["step", COL]].T.values, marker="o", label=name)
axes[0].set(title="Pérdida de entrenamiento", xlabel="paso", ylabel="pérdida")
axes[1].set(title="Val: MAE de log10 V (macro por fuente)", xlabel="paso", ylabel="MAE log10 V")
axes[1].legend(fontsize=8); plt.tight_layout(); plt.show()

In [ ]:
from camanchaca.training.runs import finished_runs, load_val_metrics
VAL = pd.DataFrame([{"modelo": r.model, "semilla": r.seed, **load_val_metrics(r.dir)}
                    for r in finished_runs(EXP_NAME, runs_root())])
VAL[[c for c in ["modelo", "semilla", "best_step", COL, "val_mae_log10", "val_mdape_pct",
                 "val_false_safe_rate", "val_ord_macro_f1"] if c in VAL]].round(4)